# EcoData · S6 — Ingesta de eventos empresariales en tiempo real
**Lanzeloth Vargas Marichi** · Proyecto Sello EcoData (alerta temprana de calidad del aire, Juliaca) · Big Data, UPeU

La guía de S6 usa *orden → pago*. Aquí se aplica el **mismo patrón** al negocio de EcoData:

| Guía S6 | EcoData | Topic | Key |
|---|---|---|---|
| `ec-orden-ms` publica `orden.creada` | **`ec-episodio-ms`** publica `aire.episodio.detectado` (PM2.5 sobre el umbral) | `ecodata-episodios` | `episodioId` |
| `ec-pago-ms` consume y publica `pago.aprobado` / `pago.rechazado` | **`ec-alerta-ms`** consume y publica `alerta.emitida` / `alerta.descartada` | `ecodata-alertas` | `episodioId` |
| evento propio (`orden.cancelada`) | **`alerta.cancelada`** (el aire volvió bajo el umbral) | `ecodata-alertas` | `episodioId` |

Los datos de entrada son **reales** (Open-Meteo/CAMS de Juliaca), no inventados. Los topics son **nuevos** (`ecodata-*`): no tocan `atmos-aire`, `atmos-clima` ni `atmos-negocio`, así que no alteran Grafana ni las alertas de Prometheus.

> Requisitos: Kafka levantado (Docker) y este cuaderno ejecutándose dentro del contenedor de Jupyter, igual que el de S12. Ejecuta las secciones **en orden** y no repitas las de publicación (duplican eventos).

**Prueba por consola (3.2 de la guía)** — hazla antes, en CMD, y captura la pantalla:
```
docker exec -it lambda26-kafka /opt/kafka/bin/kafka-topics.sh --create --topic ecodata-prueba-consola --bootstrap-server kafka:9092 --partitions 1 --replication-factor 1
docker exec -it lambda26-kafka /opt/kafka/bin/kafka-console-consumer.sh --topic ecodata-prueba-consola --bootstrap-server kafka:9092 --from-beginning
```
y en **otra** ventana CMD:
```
docker exec -it lambda26-kafka /opt/kafka/bin/kafka-console-producer.sh --topic ecodata-prueba-consola --bootstrap-server kafka:9092
```
Escribe `hola ecodata` y comprueba que aparece en el consumer.

## 0. Preparación

In [1]:
import sys, json, time
from datetime import datetime, timezone
import pandas as pd
sys.path.insert(0, "/tmp/s12/scripts")
import openmeteo_a_kafka as pub_aire

from kafka import KafkaProducer, KafkaConsumer, TopicPartition
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP = "kafka:9092"
TOPIC_EPIS, TOPIC_ALERT = "ecodata-episodios", "ecodata-alertas"
try:
    UMBRAL = float(json.load(open("/tmp/s12/modelo_pm25/meta.json"))["umbral_relativo_p95"])
except Exception:
    UMBRAL = 10.5
NIVEL_ALERTA = 15.0     # µg/m³: guía OMS 2021 para PM2.5 en 24 h
print("Umbral de episodio (p95 del entrenamiento):", UMBRAL, "µg/m³ · umbral de alerta:", NIVEL_ALERTA, "µg/m³")

def log(**kv):   # formato de la guía: cada línea declara service, component, topic y status
    print(" ".join(f"{k}={v}" for k, v in kv.items()))

Umbral de episodio (p95 del entrenamiento): 10.5 µg/m³ · umbral de alerta: 15.0 µg/m³


## 1. Crear los topics de forma explícita (3.1–3.2)
En el `compose.yml` de la guía `KAFKA_AUTO_CREATE_TOPICS_ENABLE` está en `false`: un topic no aparece solo por escribirle. Se crean con **3 particiones** para que la *key* (`episodioId`) reparta los eventos y se vea el particionado en Kafka UI.

In [2]:
adm = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
for t in (TOPIC_EPIS, TOPIC_ALERT):
    try:
        adm.create_topics([NewTopic(t, num_partitions=3, replication_factor=1)])
        print("creado:", t)
    except TopicAlreadyExistsError:
        print("ya existía:", t)
adm.close()
c = KafkaConsumer(bootstrap_servers=BOOTSTRAP)
print(sorted(t for t in c.topics() if t.startswith("ecodata")))
c.close()

creado: ecodata-episodios
creado: ecodata-alertas
['ecodata-alertas', 'ecodata-episodios']


## 2. Contratos de los eventos (3.9 y 4.1)
**`aire.episodio.detectado`** — topic `ecodata-episodios`, key `episodioId`, productor `ec-episodio-ms`, consumidor `ec-alerta-ms`
```json
{"tipoEvento":"aire.episodio.detectado","episodioId":"openmeteo-cams-juliaca-1701734400000",
 "sensorId":"openmeteo-cams-juliaca","pm25":14.2,"umbral":10.5,"origen":"openmeteo-cams","timestamp":1701734400000}
```
**`alerta.emitida` / `alerta.descartada`** — topic `ecodata-alertas`, key `episodioId`, productor `ec-alerta-ms`. Regla: PM2.5 ≥ 15 µg/m³ → *emitida*; entre 10,5 y 15 → *descartada* (episodio leve, solo informativo; equivale a `pago.rechazado` de la guía).
```json
{"tipoEvento":"alerta.emitida","episodioId":"…","sensorId":"…","pm25":18.4,"nivel":"ALERTA","motivo":"pm25>=15","origen":"ec-alerta-ms","timestamp":1701734400000}
```
**`alerta.cancelada` (evento propio, 4.1)** — topic `ecodata-alertas`, key `episodioId`, productor `ec-alerta-ms`, consumidor `ec-notificacion`. Se publica cuando, una hora después, el PM2.5 vuelve bajo el umbral.
```json
{"tipoEvento":"alerta.cancelada","episodioId":"…","sensorId":"…","pm25":7.9,"nivel":"CANCELADA","motivo":"pm25 bajo umbral en la hora siguiente","origen":"ec-alerta-ms","timestamp":1701738000000}
```
| Campo | Tipo | Unidad / regla |
|---|---|---|
| `tipoEvento` | string | uno de los tres tipos |
| `episodioId` | string | `sensorId` + hora del dato (epoch ms): idempotente |
| `pm25` | número | µg/m³, ≥ 0 |
| `umbral` | número | µg/m³ |
| `timestamp` | entero | hora del dato, epoch ms UTC |

**Por qué `episodioId` como key:** todos los eventos de un mismo episodio (detectado → emitida → cancelada) caen en la misma partición y se leen en orden entre sí, igual que `ordenId` en la guía.

## 3. Datos reales para los eventos
Se toman los episodios reales de la API de Open-Meteo (horas con PM2.5 sobre el umbral). Si la ventana reciente tiene pocos, se completa con el histórico real de Juliaca que ya está en el proyecto (`origen = openmeteo-replay`).

In [3]:
def desde_archivo(path="/tmp/s12/aq_juliaca_2023.json"):
    d = json.load(open(path)); h = d.get("hourly", d); out = []
    for t, a, b in zip(h["time"], h["pm2_5"], h["pm10"]):
        if a is None: continue
        ts = int(datetime.fromisoformat(t).replace(tzinfo=timezone.utc).timestamp() * 1000)
        out.append((ts, float(a), float(b) if b is not None else float(a)))
    return out

serie_api = pub_aire.desde_api()                      # [(ts_ms, pm25, pm10)]
pool = [(t, a, "openmeteo-cams") for t, a, b in serie_api if a > UMBRAL]
fuente = {t: a for t, a, b in serie_api}
if len(pool) < 40:
    hist = desde_archivo()
    fuente = {t: a for t, a, b in hist}
    pool = [(t, a, "openmeteo-replay") for t, a, b in hist if a > UMBRAL]
print(f"{len(pool)} episodios reales disponibles · primer episodio: {pd.to_datetime(pool[0][0], unit='ms')} UTC · PM2.5 = {pool[0][1]:.1f} µg/m³")

499 episodios reales disponibles · primer episodio: 2023-02-08 00:00:00 UTC · PM2.5 = 11.1 µg/m³


## 4. Servicio productor `ec-episodio-ms` (3.5–3.6)
Publica `aire.episodio.detectado` y registra el resultado real del broker (partición y offset), como el `whenComplete` de la guía.

In [4]:
prod = KafkaProducer(bootstrap_servers=BOOTSTRAP, acks="all",
                     key_serializer=str.encode, value_serializer=lambda v: json.dumps(v).encode())

def publicar(servicio, topic, ev, key):
    try:
        md_ = prod.send(topic, key=key, value=ev).get(timeout=10)
        log(service=servicio, component="producer", topic=md_.topic, partition=md_.partition, offset=md_.offset,
            eventType=ev["tipoEvento"], episodioId=ev["episodioId"], timestamp=ev["timestamp"], status="published")
    except Exception as e:
        log(service=servicio, component="producer", topic=topic, eventType=ev.get("tipoEvento"), status="error", error=repr(e)[:80])

def episodio(ts, pm25, origen, sensor="openmeteo-cams-juliaca"):
    return {"tipoEvento": "aire.episodio.detectado", "episodioId": f"{sensor}-{ts}", "sensorId": sensor,
            "pm25": round(float(pm25), 2), "umbral": UMBRAL, "origen": origen, "timestamp": ts}

LOTE1 = pool[:10]
for ts, a, org in LOTE1:                      # ¡ejecutar UNA sola vez!
    ev = episodio(ts, a, org); publicar("ec-episodio-ms", TOPIC_EPIS, ev, ev["episodioId"])

service=ec-episodio-ms component=producer topic=ecodata-episodios partition=2 offset=0 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1675814400000 timestamp=1675814400000 status=published
service=ec-episodio-ms component=producer topic=ecodata-episodios partition=0 offset=0 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1675818000000 timestamp=1675818000000 status=published
service=ec-episodio-ms component=producer topic=ecodata-episodios partition=2 offset=1 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1675821600000 timestamp=1675821600000 status=published
service=ec-episodio-ms component=producer topic=ecodata-episodios partition=0 offset=1 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1676592000000 timestamp=1676592000000 status=published
service=ec-episodio-ms component=producer topic=ecodata-episodios partition=1 offset=0 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-16766

## 5. Servicio `ec-alerta-ms`: consumidor y productor (3.7–3.8)
Consume `ecodata-episodios` con su propio *consumer group* (`ec-alerta-ms-group`), decide y publica el resultado en `ecodata-alertas`. **Valida cada mensaje**: uno malformado se marca `invalid` y el servicio sigue (error frecuente de 2.2 y 3.4).

In [5]:
REQ = {"tipoEvento": str, "episodioId": str, "sensorId": str, "pm25": (int, float), "umbral": (int, float),
       "origen": str, "timestamp": int}

def validar_episodio(ev):
    if not isinstance(ev, dict): return "el payload no es un objeto JSON"
    for k, tipo in REQ.items():
        if k not in ev: return f"falta el campo {k}"
        if isinstance(ev[k], bool) or not isinstance(ev[k], tipo): return f"tipo inválido en {k}"
    if ev["tipoEvento"] != "aire.episodio.detectado": return "tipoEvento inesperado"
    if ev["pm25"] < 0: return "pm25 negativo"
    return None

def servicio_alerta(espera_s=6):
    cons = KafkaConsumer(TOPIC_EPIS, bootstrap_servers=BOOTSTRAP, group_id="ec-alerta-ms-group",
                         auto_offset_reset="earliest", enable_auto_commit=True,
                         consumer_timeout_ms=int(espera_s * 1000))
    n = {"emitida": 0, "descartada": 0, "invalid": 0}
    for m in cons:
        try:
            ev = json.loads(m.value.decode("utf-8"))
            err = validar_episodio(ev)
        except Exception as e:
            ev, err = None, f"JSON ilegible ({type(e).__name__})"
        if err:
            n["invalid"] += 1
            log(service="ec-alerta-ms", component="consumer", topic=m.topic, partition=m.partition, offset=m.offset,
                status="invalid", error=f'"{err}"', rawPayload=m.value[:70])
            continue
        log(service="ec-alerta-ms", component="consumer", topic=m.topic, partition=m.partition, offset=m.offset,
            eventType=ev["tipoEvento"], episodioId=ev["episodioId"], status="received")
        emitida = ev["pm25"] >= NIVEL_ALERTA
        out = {"tipoEvento": "alerta.emitida" if emitida else "alerta.descartada", "episodioId": ev["episodioId"],
               "sensorId": ev["sensorId"], "pm25": ev["pm25"], "nivel": "ALERTA" if emitida else "INFORMATIVO",
               "motivo": f"pm25{'>=' if emitida else '<'}{NIVEL_ALERTA:g}", "origen": "ec-alerta-ms", "timestamp": ev["timestamp"]}
        publicar("ec-alerta-ms", TOPIC_ALERT, out, ev["episodioId"])
        n["emitida" if emitida else "descartada"] += 1
    cons.close()
    return n

print(servicio_alerta())

service=ec-alerta-ms component=consumer topic=ecodata-episodios partition=2 offset=0 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1675814400000 status=received
service=ec-alerta-ms component=producer topic=ecodata-alertas partition=2 offset=0 eventType=alerta.descartada episodioId=openmeteo-cams-juliaca-1675814400000 timestamp=1675814400000 status=published
service=ec-alerta-ms component=consumer topic=ecodata-episodios partition=2 offset=1 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1675821600000 status=received
service=ec-alerta-ms component=producer topic=ecodata-alertas partition=2 offset=1 eventType=alerta.descartada episodioId=openmeteo-cams-juliaca-1675821600000 timestamp=1675821600000 status=published
service=ec-alerta-ms component=consumer topic=ecodata-episodios partition=2 offset=2 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1676682000000 status=received
service=ec-alerta-ms component=producer topic=ecoda

## 6. Evento propio nuevo: `alerta.cancelada` (4.1, puntos 1–2)
Para cada alerta emitida, si **la hora siguiente** el PM2.5 real volvió bajo el umbral, `ec-alerta-ms` publica su cancelación.

In [6]:
def cancelar_alertas(lote):
    c = 0
    for ts, a, org in lote:
        if a < NIVEL_ALERTA: continue                      # solo las que se emitieron
        sig = fuente.get(ts + 3600_000)
        if sig is not None and sig < UMBRAL:
            ev = {"tipoEvento": "alerta.cancelada", "episodioId": f"openmeteo-cams-juliaca-{ts}", "sensorId": "openmeteo-cams-juliaca",
                  "pm25": round(float(sig), 2), "nivel": "CANCELADA", "motivo": "pm25 bajo umbral en la hora siguiente",
                  "origen": "ec-alerta-ms", "timestamp": ts + 3600_000}
            publicar("ec-alerta-ms", TOPIC_ALERT, ev, ev["episodioId"]); c += 1
    return c
n_canc = cancelar_alertas(LOTE1)
print("cancelaciones publicadas:", n_canc)
if n_canc == 0:
    print("Ninguna alerta emitida de este lote bajó del umbral a la hora siguiente. Prueba con un lote distinto: cancelar_alertas(pool[10:40]) (una sola vez).")

cancelaciones publicadas: 0
Ninguna alerta emitida de este lote bajó del umbral a la hora siguiente. Prueba con un lote distinto: cancelar_alertas(pool[10:40]) (una sola vez).


### 6.1 Segundo consumidor sobre el mismo topic
`ec-notificacion` lee `ecodata-alertas` con **otro consumer group**: recibe una copia completa de los eventos sin quitárselos a nadie (responde "¿por qué Kafka no borra un evento al leerlo?").

In [16]:
cons = KafkaConsumer(TOPIC_ALERT, bootstrap_servers=BOOTSTRAP, group_id="ec-notificacion-group",
                     auto_offset_reset="earliest", consumer_timeout_ms=5000)
filas = []
for m in cons:
    ev = json.loads(m.value); filas.append((m.partition, m.offset, ev["tipoEvento"], ev["episodioId"][-13:], ev["pm25"]))
cons.close()
df = pd.DataFrame(filas, columns=["partition", "offset", "tipoEvento", "hora_ms", "pm25"])
print(df.tipoEvento.value_counts().to_string()); df.sort_values(["partition", "offset"]).head(12)

tipoEvento
alerta.descartada    30
alerta.emitida        3
alerta.cancelada      3


,partition,offset,tipoEvento,hora_ms,pm25
0,0,5,alerta.descartada,1676782800000,13.1
1,0,6,alerta.descartada,1676786400000,11.9
2,0,7,alerta.descartada,1676793600000,11.3
3,0,8,alerta.descartada,1677402000000,10.7
4,0,9,alerta.descartada,1677409200000,12.1
5,0,10,alerta.descartada,1678258800000,10.7
6,0,11,alerta.descartada,1681120800000,10.8
7,0,12,alerta.descartada,1681444800000,10.7
8,0,13,alerta.emitida,1682074800000,16.1
9,0,14,alerta.emitida,1693998000000,21.0


In [15]:
# Episodios con PM2.5 >= nivel de alerta, y cuya hora siguiente baje del umbral (para que se pueda cancelar)
ALTOS = [x for x in pool
         if x[1] >= NIVEL_ALERTA
         and fuente.get(x[0] + 3600_000) is not None
         and fuente[x[0] + 3600_000] < UMBRAL][:5]
print(len(ALTOS), "episodios con PM2.5 >=", NIVEL_ALERTA, "y bajada a la hora siguiente")
for ts, a, org in ALTOS:
    ev = episodio(ts, a, org)
    publicar("ec-episodio-ms", TOPIC_EPIS, ev, ev["episodioId"])
print(servicio_alerta())
print("cancelaciones publicadas:", cancelar_alertas(ALTOS))

3 episodios con PM2.5 >= 15.0 y bajada a la hora siguiente
service=ec-episodio-ms component=producer topic=ecodata-episodios partition=0 offset=13 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1682074800000 timestamp=1682074800000 status=published
service=ec-episodio-ms component=producer topic=ecodata-episodios partition=2 offset=10 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1692453600000 timestamp=1692453600000 status=published
service=ec-episodio-ms component=producer topic=ecodata-episodios partition=0 offset=14 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1693998000000 timestamp=1693998000000 status=published
service=ec-alerta-ms component=consumer topic=ecodata-episodios partition=2 offset=10 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1692453600000 status=received
service=ec-alerta-ms component=producer topic=ecodata-alertas partition=2 offset=10 eventType=alerta.emitida episodioId=open

## 7. Mensaje que incumple el contrato (4.1, punto 3)
Se publican a propósito dos mensajes malos en `ecodata-episodios`: texto plano (como el `hola kafka` de la consola) y un JSON **sin `episodioId`**. El consumidor no debe caerse.

In [10]:
prod = KafkaProducer(bootstrap_servers=BOOTSTRAP, acks="all",
                     key_serializer=str.encode,
                     value_serializer=lambda v: v if isinstance(v, bytes) else json.dumps(v).encode())

prod.send(TOPIC_EPIS, key="malo-1", value=b"hola kafka (texto plano)")
prod.send(TOPIC_EPIS, key="malo-2",
          value=json.dumps({"tipoEvento": "aire.episodio.detectado", "sensorId": "openmeteo-cams-juliaca",
                            "pm25": 18.0, "umbral": UMBRAL, "origen": "prueba", "timestamp": 1}).encode())
prod.flush()
print(servicio_alerta())      # debe mostrar status=invalid x2 y seguir funcionando

service=ec-alerta-ms component=consumer topic=ecodata-episodios partition=1 offset=2 status=invalid error="JSON ilegible (JSONDecodeError)" rawPayload=b'hola kafka (texto plano)'
service=ec-alerta-ms component=consumer topic=ecodata-episodios partition=1 offset=3 status=invalid error="falta el campo episodioId" rawPayload=b'{"tipoEvento": "aire.episodio.detectado", "sensorId": "openmeteo-cams-'
{'emitida': 0, 'descartada': 0, 'invalid': 2}


## 8. Lag del *consumer group* antes y después de una ráfaga (4.1, punto 4)
*Lag* = mensajes publicados que el grupo todavía no leyó. Se mide con la API y se compara con Kafka UI (Topics → `ecodata-episodios` → Consumers) y con la consola:
```
docker exec -it lambda26-kafka /opt/kafka/bin/kafka-consumer-groups.sh --bootstrap-server kafka:9092 --describe --group ec-alerta-ms-group
```

In [12]:
def lag(group, topic):
    # consumidor con el mismo group_id, sin suscribirse: solo consulta el offset confirmado (no altera el grupo)
    c = KafkaConsumer(bootstrap_servers=BOOTSTRAP, group_id=group, enable_auto_commit=False)
    tps = [TopicPartition(topic, p) for p in sorted(c.partitions_for_topic(topic))]
    fin = c.end_offsets(tps)
    filas = []
    for t in tps:
        cur = c.committed(t) or 0
        filas.append((t.partition, cur, fin[t], fin[t] - cur))
    c.close()
    return pd.DataFrame(filas, columns=["partición", "offset_leído", "offset_final", "lag"])
print("ANTES de la ráfaga"); a = lag("ec-alerta-ms-group", TOPIC_EPIS); display(a); print("lag total:", a.lag.sum())

ANTES de la ráfaga


,partición,offset_leído,offset_final,lag
0,0,5,5,0
1,1,4,4,0
2,2,3,3,0


lag total: 0


In [13]:
RAFAGA = pool[10:40]                                   # ¡ejecutar UNA sola vez!
for ts, a_, org in RAFAGA:
    ev = episodio(ts, a_, org); prod.send(TOPIC_EPIS, key=ev["episodioId"], value=ev)
prod.flush()
print(f"ráfaga publicada: {len(RAFAGA)} eventos")
print("DESPUÉS de publicar, SIN consumir"); b = lag("ec-alerta-ms-group", TOPIC_EPIS); display(b); print("lag total:", b.lag.sum())

ráfaga publicada: 30 eventos
DESPUÉS de publicar, SIN consumir


,partición,offset_leído,offset_final,lag
0,0,5,13,8
1,1,4,19,15
2,2,3,10,7


lag total: 30


In [14]:
print(servicio_alerta())
print("DESPUÉS de consumir"); d = lag("ec-alerta-ms-group", TOPIC_EPIS); display(d); print("lag total:", d.lag.sum())

service=ec-alerta-ms component=consumer topic=ecodata-episodios partition=2 offset=3 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1676779200000 status=received
service=ec-alerta-ms component=producer topic=ecodata-alertas partition=2 offset=3 eventType=alerta.descartada episodioId=openmeteo-cams-juliaca-1676779200000 timestamp=1676779200000 status=published
service=ec-alerta-ms component=consumer topic=ecodata-episodios partition=2 offset=4 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1676804400000 status=received
service=ec-alerta-ms component=producer topic=ecodata-alertas partition=2 offset=4 eventType=alerta.descartada episodioId=openmeteo-cams-juliaca-1676804400000 timestamp=1676804400000 status=published
service=ec-alerta-ms component=consumer topic=ecodata-episodios partition=2 offset=5 eventType=aire.episodio.detectado episodioId=openmeteo-cams-juliaca-1677405600000 status=received
service=ec-alerta-ms component=producer topic=ecoda

,partición,offset_leído,offset_final,lag
0,0,13,13,0
1,1,19,19,0
2,2,10,10,0


lag total: 0


### Qué capturar
1. CMD con la prueba por consola (topic `ecodata-prueba-consola`).
2. Kafka UI → Topics: `ecodata-episodios` y `ecodata-alertas` con 3 particiones y mensajes visibles.
3. Kafka UI → un mensaje abierto de cada topic.
4. Las salidas de las secciones 4, 5 y 6 (logs `component=producer` / `consumer`).
5. La sección 7 con `status=invalid` y el consumidor siguiendo.
6. Las tres tablas de lag de la sección 8 (antes, sin consumir, después).

Si algo falla, **no repitas** las secciones de publicación: avísame con la captura.

In [ ]:
prod.close()